<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
import pandas as pd
import os
import subprocess
import numpy as np
import duckdb
from google.colab import userdata

In [13]:

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":      f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":      f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily_march": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

FEAT_START, FEAT_END, OUT_START = "2026-03-01", "2026-03-15", "2026-03-16"


In [26]:

features = con.sql(f"""
    SELECT
        client_hash_id AS client_id,
        content_hash_id AS content_id,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,
        AVG(gsc_avg_position) FILTER (WHERE gsc_impressions > 0) AS avg_position,
        SUM(gsc_clicks) * 100.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM {TABLES['fact_daily_march']}
    WHERE report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()

outcome = con.sql(f"""
    SELECT client_hash_id AS client_id, content_hash_id AS content_id,
           SUM(gsc_impressions) AS impressions_outcome
    FROM {TABLES['fact_daily_march']}
    WHERE report_date >= DATE '{OUT_START}'
    GROUP BY 1, 2
""").df()

df = features.merge(outcome, on=["client_id", "content_id"], how="left")
df["impressions_outcome"] = df["impressions_outcome"].fillna(0)
df["is_declining_label"] = (df["impressions_outcome"] < 0.80 * df["impressions_90d"]).astype(int)

df["avg_position"] = df["avg_position"].fillna(0)
df["avg_position_missing"] = (df["avg_position"] == 0).astype(int)

content_meta = con.sql(f"""
    SELECT content_hash_id AS content_id, content_updated_date, word_count
    FROM {TABLES['dim_content']}
""").df()
df = df.merge(content_meta, on="content_id", how="left")



In [27]:
print(f"rows: {len(df)}, clients: {df['client_id'].nunique()}, base decline rate: {df['is_declining_label'].mean():.3f}")
df.head()

rows: 120513, clients: 41, base decline rate: 0.296


,client_id,content_id,impressions_90d,clicks_90d,avg_position,ctr,impressions_outcome,is_declining_label,avg_position_missing,content_updated_date,word_count
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,57.0,0.0,3.659683,0.000000,20.0,1,0,2026-06-29,3579
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,199.0,2.0,4.086084,1.005025,403.0,0,0,2026-06-29,2455
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,467.0,1.0,4.449176,0.214133,343.0,1,0,2026-06-29,3653
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,56.0,0.0,6.600595,0.000000,26.0,1,0,2026-06-29,3096
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,771.0,1.0,1.883472,0.129702,1087.0,0,0,2026-06-29,3305


In [28]:
df["days_since_update"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(df["content_updated_date"])).dt.days

tier_order = ["0-30", "31-90", "91-180", "181+", "unknown"]
df["freshness_tier"] = pd.cut(
    df["days_since_update"].where(df["days_since_update"] >= 0),
    bins=[-1, 30, 90, 180, float("inf")],
    labels=tier_order[:-1]
).astype("object")
df["freshness_tier"] = df["freshness_tier"].fillna("unknown")
df["freshness_tier_enc"] = df["freshness_tier"].map({t: i for i, t in enumerate(tier_order)})
df["freshness_unknown"] = (df["freshness_tier"] == "unknown").astype(int)

In [36]:
print("Share of rows updated after the feature window:", (df["content_updated_date"] > "2026-03-15").mean())

Share of rows updated after the feature window: 0.8158621891414204


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**RULE:**

> I want to pull pages where performance is dropping off compared to their overall search visibility. Basically, I'm flagging content that's either getting old or ranking in the Top 20 but failing to turn those impressions into actual clicks. I also set up a high-priority "High Stale Decoy" bucket — pages that haven't been touched in forever but still draw massive search volume. Those are our highest-risk/highest-reward updates.


**Reason Codes:**

HIGH_STALE_DECOY: Older 6 months old (181+) AND top-tier impressions (>5,000 90d).

CTR_UNDERPERFORM: Sitting in the Top 20 with a CTR under 0.5%.

STALE_CONTENT: Content hasn't been updated in over 90 days.

COMBO_STALE_CTR: Triggered both the staleness and low-CTR flags.

**Why I Chose These Thresholds**

The baseline uses simple thresholds so that the ranking can be explained to an editor without requiring a machine-learning model.

- I used a 90-day window because it provides a reasonable period for identifying pages that may have become outdated.
- I used a CTR threshold of 0.5% because pages ranking in the top 20 would generally be expected to receive some clicks. A very low CTR can indicate that the search result is not attracting users even when the page ranks reasonably well.
- I required at least 500 impressions so that very low-traffic pages do not trigger the CTR rule based on noisy CTR estimates.
- freshness_tier is unknown for about 82% of rows because dim_content only stores current-state update dates rather than a complete update history. These pages are therefore handled separately through the freshness_unknown signal rather than being treated as recently or historically updated pages.

These thresholds are therefore best viewed as practical baseline rules rather than statistically optimal cutoffs.

In [15]:
# Signal 1: staleness (behind FlyRank's refresh flags) ---
order = ["0-30", "31-90", "91-180", "181+"]
staleness_table = (df.groupby("freshness_tier")
                      .agg(n=("is_declining_label", "size"),
                           decline_rate=("is_declining_label", "mean"))
                      .reindex(order))
print(staleness_table)

                    n  decline_rate
freshness_tier                     
0-30              204      0.455882
31-90           21668      0.371793
91-180            285      0.371930
181+               45      0.488889


/tmp/ipykernel_1014/2752671077.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_table = (df.groupby("freshness_tier")


VERDICT: MIXED

The decline rate is lower for the 31–90 and 91–180 day groups than for the 0–30 day group, while the 181+ group has the highest decline rate. However, the 181+ bucket contains only 45 pages, compared with more than 21,000 pages in the 31–90 bucket. Because the sample sizes are very uneven, I would not treat the differences between the freshness groups as strong evidence on their own.

I kept freshness as part of the baseline because it is an intuitive and explainable signal: pages that have not been updated recently may need attention. However, the results show that freshness alone is not a reliable indicator of decline across all pages

In [30]:
# Signal 2: CTR-vs-position (behind the CTR-fix logic) ---
df["low_ctr_flag"] = (
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) & (df["avg_position"] <= 20) &
    (df["ctr"] < 0.5)
)

ctr_table = (df.groupby("low_ctr_flag")
               .agg(n=("is_declining_label", "size"),
                    decline_rate=("is_declining_label", "mean")))
print(ctr_table)



                  n  decline_rate
low_ctr_flag                     
False         93930      0.300053
True          26583      0.283903


VERDICT: NOT CONFIRMED

In this version of the data, pages flagged for low CTR actually have a slightly lower decline rate than pages that are not flagged. The difference is small, so the current output does not support treating the low-CTR rule as a strong standalone signal of decline.

I still kept the CTR rule in the baseline because CTR can be useful when combined with other signals, especially for pages that rank well but receive very few clicks. However, this analysis shows why the rule should not be interpreted as proof that a page is declining.

I also did not want pages with only a few impressions to dominate the results. A CTR calculated from 20 or 30 impressions can fluctuate substantially, so I required at least 500 impressions before applying the CTR rule.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [31]:

# Thresholds based on Signal Audit
STALE_TIERS = ["91-180", "181+"]
CTR_THRESHOLD = 0.5  # 0.5%
IMPRESSION_DECOY_LEVEL = 5000

# 1. Flag components
df["stale_flag"] = df["freshness_tier"].isin(STALE_TIERS)
df["low_ctr_flag"] = (df["avg_position"] <= 20) & (df["ctr"] < CTR_THRESHOLD) & (df["impressions_90d"] >= 500)
df["is_decoy"] = (df["freshness_tier"] == "181+") & (df["impressions_90d"] >= IMPRESSION_DECOY_LEVEL)

# 2. Score logic (Weights: Decoy = 3, Combo = 2, Single = 1)
def calculate_score(row):
    if row["is_decoy"]: return 3
    if row["stale_flag"] and row["low_ctr_flag"]: return 2
    if row["stale_flag"] or row["low_ctr_flag"]: return 1
    return 0

df["score"] = df.apply(calculate_score, axis=1)

# 3. Reason code implementation (Ensures alignment with Section 1)
def get_reason(row):
    if row["is_decoy"]: return "HIGH_STALE_DECOY"
    if row["stale_flag"] and row["low_ctr_flag"]: return "COMBO_STALE_CTR"
    if row["low_ctr_flag"]: return "CTR_UNDERPERFORM"
    if row["stale_flag"]: return "STALE_CONTENT"
    return "NO_SIGNAL"

df["reason_code"] = df.apply(get_reason, axis=1)

# 4. Action labels
df["action"] = df["score"].apply(lambda s: "REFRESH_NOW" if s >= 2 else ("REVIEW" if s == 1 else "NO_ACTION"))

# Rank by score first, then impressions to prioritize 'big' pages
queue = df.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = queue.index + 1

# Export
os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id", "client_id", "rank", "score", "reason_code", "action", "is_declining_label"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Queue Size: {len(queue)} | Baseline Decline Rate: {df['is_declining_label'].mean():.2%}")

Queue Size: 120513 | Baseline Decline Rate: 29.65%


In [18]:
!head work/outputs/baseline_action_score.csv

content_id,client_id,rank,score,reason_code,action,is_declining_label
content_f8e84719d52ab1fe,client_20259bd6705d81d4,1,2,COMBO_STALE_CTR,REFRESH_NOW,1
content_bea86ce3455100b0,client_c182d11e4862a37d,2,2,COMBO_STALE_CTR,REFRESH_NOW,1
content_5120dcbbb086843d,client_c182d11e4862a37d,3,2,COMBO_STALE_CTR,REFRESH_NOW,1
content_a96ce61bf37c8ac7,client_20259bd6705d81d4,4,2,COMBO_STALE_CTR,REFRESH_NOW,0
content_9c057b66c30a3abb,client_73cda7b4e4f265ea,5,1,CTR_UNDERPERFORM,REVIEW,1
content_8e1334d6356668e3,client_73cda7b4e4f265ea,6,1,CTR_UNDERPERFORM,REVIEW,0
content_945d6ff91386c817,client_62f4a7e64f5e0096,7,1,CTR_UNDERPERFORM,REVIEW,1
content_0c5606abaaab3178,client_62f4a7e64f5e0096,8,1,CTR_UNDERPERFORM,REVIEW,1
content_097459d155cccb26,client_20259bd6705d81d4,9,1,STALE_CONTENT,REVIEW,0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [19]:
# Code to display top 20 for review
review_cols = ["content_id", "rank", "score", "reason_code", "action",
               "freshness_tier", "avg_position", "ctr", "impressions_90d"]
print(queue[review_cols].head(20).to_string(index=False))

              content_id  rank  score      reason_code      action freshness_tier  avg_position      ctr  impressions_90d
content_f8e84719d52ab1fe     1      2  COMBO_STALE_CTR REFRESH_NOW         91-180     11.414193 0.000000           2133.0
content_bea86ce3455100b0     2      2  COMBO_STALE_CTR REFRESH_NOW           181+      5.687620 0.000000           2043.0
content_5120dcbbb086843d     3      2  COMBO_STALE_CTR REFRESH_NOW           181+      6.998111 0.000000           1413.0
content_a96ce61bf37c8ac7     4      2  COMBO_STALE_CTR REFRESH_NOW         91-180     19.255662 0.000000            770.0
content_9c057b66c30a3abb     5      1 CTR_UNDERPERFORM      REVIEW          31-90      8.607910 0.000000          83772.0
content_8e1334d6356668e3     6      1 CTR_UNDERPERFORM      REVIEW            NaN      4.579049 0.001708          58553.0
content_945d6ff91386c817     7      1 CTR_UNDERPERFORM      REVIEW            NaN      6.413782 0.004056          49314.0
content_0c5606abaaab3178

### Top-20 Review (Data-Bound SME Critique)

| Rank | Content ID | Reason Code | Key Metrics | Why it got flagged | What would make this wrong? (Data-Bound Limits) |
|------|------------|-------------|-------------|--------------------|--------------------------------------------------|
| 1 | `content_f8e84719d52ab1fe` | COMBO_STALE_CTR | 2.1k Imps, 0.00% CTR, Pos 11.4, 91–180 tier | Stale page combined with CTR below the 0.5% threshold. | Position 11.4 is already around the Page 2 boundary, so low CTR may partly reflect ranking position rather than content decay. |
| 2 | `content_bea86ce3455100b0` | COMBO_STALE_CTR | 2.0k Imps, 0.00% CTR, Pos 5.7, 181+ tier | Very old page with a strong average position and zero recorded CTR. | The available data cannot show whether the zero CTR is caused by poor search-result appeal, query intent, SERP features, or a measurement issue. |
| 3 | `content_5120dcbbb086843d` | COMBO_STALE_CTR | 1.4k Imps, 0.00% CTR, Pos 7.0, 181+ tier | Old page with a top-10 average position and zero recorded CTR. | Without query-level or SERP data, we cannot determine why the page receives no recorded clicks or whether a content refresh would help. |
| 4 | `content_a96ce61bf37c8ac7` | COMBO_STALE_CTR | 770 Imps, 0.00% CTR, Pos 19.3, 91–180 tier | Stale page combined with very low CTR. | Position 19.3 is deep on Page 2, so low CTR may be explained by limited visibility rather than content decay. |
| 5 | `content_9c057b66c30a3abb` | CTR_UNDERPERFORM | 83.8k Imps, 0.00% CTR, Pos 8.6, 31–90 tier | Very high impressions and a top-10 average position but zero recorded CTR. | This page is not stale, so the flag depends on CTR alone. The current data cannot tell whether the low CTR comes from content, query intent, SERP features, or measurement. |
| 6 | `content_8e1334d6356668e3` | CTR_UNDERPERFORM | 58.6k Imps, 0.17% CTR, Pos 4.6, Freshness unknown | Strong search visibility with CTR well below the 0.5% threshold. | `freshness_tier` is unknown, so we cannot determine whether the page is actually stale. Query-level information is also unavailable. |
| 7 | `content_945d6ff91386c817` | CTR_UNDERPERFORM | 49.3k Imps, 0.41% CTR, Pos 6.4, Freshness unknown | Top-10 average position with CTR below the 0.5% cutoff. | The aggregate Signal 2 analysis did not confirm low CTR as a strong standalone decline signal, so this flag should be treated as a review cue rather than proof of decay. |
| 8 | `content_0c5606abaaab3178` | CTR_UNDERPERFORM | 27.7k Imps, 0.00% CTR, Pos 5.4, Freshness unknown | High impressions and strong average position but zero recorded CTR. | The data cannot distinguish genuine zero-click behaviour from a tracking/data-quality issue or query-level effects. Freshness is also unavailable. |
| 9 | `content_097459d155cccb26` | STALE_CONTENT | 20.3k Imps, 19.2% CTR, Pos 12.6, 91–180 tier | The page is in the 91–180 day freshness tier and therefore triggers the staleness rule. | Its CTR is relatively high, so staleness alone does not show that the page is underperforming. A refresh may have little benefit if the content is still satisfying users. |
| 10 | `content_d61fc394d10cba41` | CTR_UNDERPERFORM | 18.6k Imps, 0.00% CTR, Pos 3.4, Freshness unknown | Very strong average position but zero recorded CTR. | Position 3.4 makes the result unusual, but the available features cannot explain whether this is caused by content, query intent, SERP behaviour, or measurement. |
| 11 | `content_c9f840183215651b` | CTR_UNDERPERFORM | 18.4k Imps, 0.00% CTR, Pos 5.2, Freshness unknown | Top-10 visibility with zero recorded CTR. | The baseline cannot determine the reason for the zero CTR because query-level and SERP-level information are not available. |
| 12 | `content_22588e765b93dfac` | CTR_UNDERPERFORM | 17.2k Imps, 0.00% CTR, Pos 8.2, 31–90 tier | Good average position but no recorded clicks. | The page is not in a stale tier, so this flag is entirely driven by CTR. The data does not establish that rewriting the content would improve performance. |
| 13 | `content_f2df5a8a9057783e` | STALE_CONTENT | 17.0k Imps, 27.7% CTR, Pos 16.2, 91–180 tier | The page is flagged because it falls into the 91–180 freshness tier. | Its CTR is relatively strong, so there is limited evidence that freshness is causing a performance problem. Its average position may also be a more important limitation. |
| 14 | `content_23a42776a7009b65` | CTR_UNDERPERFORM | 16.7k Imps, 0.00% CTR, Pos 8.7, Freshness unknown | Top-10 average position with zero recorded CTR. | Missing freshness information limits the analysis, and the available features cannot identify whether the low CTR is caused by content or by query/SERP factors. |
| 15 | `content_66d1fffc91f4f029` | STALE_CONTENT | 15.1k Imps, 19.2% CTR, Pos 12.6, 91–180 tier | Staleness alone triggered the review. | The 19.2% CTR suggests the page is receiving clicks, so the stale flag does not by itself prove that the content needs updating. |
| 16 | `content_8d395745c4d76f9e` | CTR_UNDERPERFORM | 15.0k Imps, 0.00% CTR, Pos 3.9, Freshness unknown | Strong average position but zero recorded CTR. | The combination is unusual, but there is not enough information to determine whether it reflects content performance, query intent, SERP behaviour, or a data issue. |
| 17 | `content_b956947c822af734` | STALE_CONTENT | 14.8k Imps, 3.37% CTR, Pos 38.8, 91–180 tier | The page is in the 91–180 freshness tier and is therefore flagged as stale. | Position 38.8 is very deep in the rankings, so poor visibility may explain its performance more than freshness. A content refresh may not solve the ranking problem. |
| 18 | `content_454d3e5e51175aff` | CTR_UNDERPERFORM | 13.6k Imps, 0.00% CTR, Pos 6.4, Freshness unknown | Top-10 visibility with zero recorded CTR. | Freshness is unavailable, and the current data cannot show whether the zero CTR is caused by the content, search intent, SERP features, or measurement. |
| 19 | `content_1ff6231687184dec` | CTR_UNDERPERFORM | 12.6k Imps, 0.00% CTR, Pos 16.7, Freshness unknown | High impressions combined with CTR below the threshold. | Position 16.7 is already on Page 2, so low CTR may be largely explained by limited visibility rather than content decay. |
| 20 | `content_9e0a8a913953b8d3` | CTR_UNDERPERFORM | 12.0k Imps, 0.00% CTR, Pos 8.5, Freshness unknown | Strong average position but zero recorded CTR. | The output does not include query-level CTR, SERP features, or page metadata, so it cannot establish why the page receives no recorded clicks. |

**Key observation:** Several high-priority pages have substantial impression volume and a CTR of exactly 0.00%, including pages with average positions between roughly 3 and 9. This is an unusual pattern worth investigating, but the available data does not prove that it is caused by a tracking problem. It could also reflect query mix, SERP features, or other factors that are not included in the baseline.

Another limitation is the large number of pages with `freshness_tier` missing. These pages should be treated as having **unknown freshness**, not as fresh or stale.

The review also shows two important weaknesses in the baseline. Some pages are flagged as stale even though they have relatively strong CTR, while others are flagged for low CTR even when their ranking position is deep enough that low CTR may be expected. This supports using the baseline as a prioritisation benchmark rather than treating every flag as proof that a page needs a refresh.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [34]:
print("Columns used in scoring: freshness_tier, avg_position, impressions_90d, ctr")
print("impressions_outcome used only to build is_declining_label (the label) —",
      "never inside stale_flag, low_ctr_flag, score, reason_code, or action.")

borderline = queue[queue["score"] == 1].sort_values("impressions_90d").head(3)
print(borderline[review_cols])

Columns used in scoring: freshness_tier, avg_position, impressions_90d, ctr
impressions_outcome used only to build is_declining_label (the label) — never inside stale_flag, low_ctr_flag, score, reason_code, or action.
                     content_id   rank  score    reason_code  action  \
26885  content_18498fdc015f69b2  26886      1  STALE_CONTENT  REVIEW   
26860  content_be509a84f58075bb  26861      1  STALE_CONTENT  REVIEW   
26861  content_c8dc2f36416869e5  26862      1  STALE_CONTENT  REVIEW   

      freshness_tier  avg_position  ctr  impressions_90d  
26885         91-180      6.523810  0.0             10.0  
26860         91-180      8.266667  0.0             10.0  
26861         91-180      8.125000  0.0             10.0  


In [21]:
# Section 4: Inspecting low-volume flagged rows (score == 1)
borderline_flagged = queue[queue["score"] == 1].sort_values("impressions_90d").head(3)
print(borderline_flagged[["content_id", "rank", "score", "reason_code", "impressions_90d", "avg_position", "ctr", "freshness_tier"]])

                    content_id  rank  score    reason_code  impressions_90d  \
5396  content_1fb68d1cecea9f03  5397      1  STALE_CONTENT             10.0   
5411  content_c91f65aa4ec83b8e  5412      1  STALE_CONTENT             10.0   
5412  content_06735fffc203e7af  5413      1  STALE_CONTENT             10.0   

      avg_position  ctr freshness_tier  
5396      2.200000  0.0         91-180  
5411      6.305556  0.0           181+  
5412      4.875000  0.0         91-180  


In [35]:
borderline_flagged

,client_id,content_id,impressions_90d,clicks_90d,avg_position,ctr,impressions_outcome,is_declining_label,avg_position_missing,content_updated_date,...,days_since_update,freshness_tier,freshness_tier_enc,low_ctr_flag,stale_flag,is_decoy,score,reason_code,action,rank
5396,client_3ffa76342f366962,content_1fb68d1cecea9f03,10.0,0.0,2.200000,0.0,5.0,1,0,2025-11-10,...,141,91-180,2,False,True,False,1,STALE_CONTENT,REVIEW,5397
5411,client_b10cb2997d0c7c86,content_c91f65aa4ec83b8e,10.0,0.0,6.305556,0.0,8.0,0,0,2025-08-12,...,231,181+,3,False,True,False,1,STALE_CONTENT,REVIEW,5412
5412,client_157ffe4d4a595515,content_06735fffc203e7af,10.0,0.0,4.875000,0.0,3.0,1,0,2025-12-08,...,113,91-180,2,False,True,False,1,STALE_CONTENT,REVIEW,5413


### Weak Picks & Borderline Analysis

The borderline review found three particularly weak baseline picks: `content_1fb68d1cecea9f03` has only 10 impressions and an average position of 2.2, `content_c91f65aa4ec83b8e` has 10 impressions and an average position of 6.3, and `content_06735fffc203e7af` has 10 impressions and an average position of 4.9. All three are flagged as `STALE_CONTENT` with score 1, even though each has only 10 impressions and 0 clicks, while `low_ctr_flag` is False because they do not meet the 500-impression requirement.

This exposes a weakness in the baseline: the staleness rule has no minimum impression threshold, so pages with almost no traffic can still enter the review queue based only on age. For example, `content_1fb68d1cecea9f03` averages position 2.2 despite having only 10 impressions, so the available data does not provide enough evidence to conclude that its stale status represents a meaningful content problem or that refreshing it would have useful impact.

**Leakage Check**

I checked that the baseline does not use the outcome information it is supposed to predict.

impressions_outcome is strictly omitted from the baseline scoring logic, along with the target label. The baseline relies only on information available for the page when the queue is constructed, such as freshness, position, impressions, and CTR.

This prevents the baseline from using future outcome information to give a page a higher score and keeps the comparison with the machine-learning models fair.

**Conclusion**

This baseline uses two simple, explainable signals: content freshness and CTR performance. When audited individually, neither turned out to be a strong standalone predictor — freshness showed mixed results across uneven sample sizes, and low CTR was not confirmed as linked to higher decline rates on its own. The scoring logic remains easy to interpret and leak-free, and it still gives a transparent starting point for comparison. The main value of this baseline is as an honest benchmark to beat, not as a proven detection rule — which is exactly what the ML model in the next sprint needs to improve on.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.